In [1]:
import numpy as np
import torch
import random

from Conv1DAE import Conv1DAE, detect_anomalies_conv1dae, conv1dae_train
from optimize_models import latency_to_detection, get_basic_metrics, train_test_split_anomaly_sequence, optimize_conv1dae
from prepare_data import load_or_cache_mit_bih, load_or_cache_bonn, \
    get_mit_bih_segments, get_bonn_segments
from visualizations import heatmaps, segments_reconstruction

torch.manual_seed(0)
random.seed(0)

#### <center>Zbiór EEG Bonn</center>

Uznajemy, że zbiór E to outliery, a reszta:
- zdrowi oczy otwarte -> A,
- zdrowi oczy zamknięte -> B,
- pacjenci między napadami (zdrowa półkula) -> C,
- pacjenci między napadami (strefa padaczkowa) -> D,

są zdrowi.

In [2]:
eeg_bonn_dataset = load_or_cache_bonn()

Przygotowany zbiór EEG-Bonn

Każda sekwencja ma przypisaną etykietę na podstawie przynależności do zbioru. Etykieta segmentu jest przypisana na podstawie stosunku liczby anomalii do normalnych próbek na poziomie sekwencji.

In [3]:
bonn_overlaps = [0.25, 0.5, 0.75]
bonn_window_sizes = [2, 4, 6]

<center>Eksperymenty dla 1D Conv-AE</center>

In [4]:
conv1dae_bonn_experiments, conv1dae_bonn_heatmap, conv1dae_bonn_training_params = [
    {
        latent: {
            s: {
                o: None for o in bonn_overlaps
            } for s in bonn_window_sizes
        } for latent in [2, 4]
    }  for _ in range(3)
]

for latent, seconds in conv1dae_bonn_experiments.items():
    for second, overlaps in seconds.items():
        for overlap in overlaps.keys():
            # Przygotowanie danych
            X, y, _ = get_bonn_segments(eeg_bonn_dataset, second, overlap)

            # Podział na train/test
            X_train, X_test, y_train, y_test = train_test_split_anomaly_sequence(X, y, random_state=42)

            # Optymalizacja parametrów Conv1DAE
            conv1dae_study = optimize_conv1dae(
                X=X_train,
                y=y_train,
                latent=latent,
                dataset_name="EEG Bonn",
                n_trials=10
            )
            conv1dae_params = conv1dae_study.best_params
            conv1dae_train_params = {k: conv1dae_params[k] for k in ["lr", "epochs", "alpha", "beta", "gamma"]}
            conv1dae_loss_params = {k: conv1dae_params[k] for k in ["alpha", "beta", "gamma"]}

            conv1dae_bonn_training_params[latent][second][overlap] = conv1dae_params

            # Konwersja z numpy do torch
            X_train, X_test = torch.from_numpy(X_train).to(dtype=torch.float32, device='cuda'), torch.from_numpy(X_test).to(dtype=torch.float32, device='cuda')

            # Autoenkoder
            conv1dae = Conv1DAE(input_dim=1, latent_dim=latent)
            X_train = X_train[:, :, None, :]
            X_test = X_test[:, :, None, :]

            # Trening
            conv1dae_train(
                model=conv1dae,
                data=X_train,
                **conv1dae_train_params
            )

            # Wyznaczenie progu z danych treningowych
            y_train_scores, reconstruction, encoded = detect_anomalies_conv1dae(
                model=conv1dae,
                data=X_train,
                **conv1dae_loss_params
            )
            train_threshold = np.percentile(y_train_scores, 99)

            # Wyznaczenie etykiet -1/1 na zbiorze testowym
            y_test_scores, reconstruction, encoded = detect_anomalies_conv1dae(
                model=conv1dae,
                data=X_test,
                **conv1dae_loss_params
            )
            y_pred = np.where(y_test_scores > train_threshold, -1, 1)

            # Zapisanie metryk
            experiment_basic_metrics = get_basic_metrics(
                y_true=y_test.reshape(-1, 1).squeeze(),
                y_pred=y_pred.reshape(-1, 1).squeeze(),
                y_scores=y_test_scores.reshape(-1, 1).squeeze()
            )
            conv1dae_test_latencies = latency_to_detection(y_test, y_pred)
            conv1dae_bonn_experiments[latent][second][overlap] = {**experiment_basic_metrics, **conv1dae_test_latencies}
            conv1dae_bonn_heatmap[latent][second][overlap] = y_test_scores

            # Rekonstrukcja sygnałów
            segments_reconstruction(
                X_test=X_test.cpu().numpy().squeeze(2),
                X_pred=reconstruction,
                y_true=y_test,
                y_pred=y_pred.squeeze(),
                title="",
                save_path=f"{latent}_{second}_{int(overlap * 100)}_reconstruction_analysis_bonn.png"
            )

[I 2026-01-27 21:23:52,800] A new study created in memory with name: Optuna for Conv1DAE on EEG Bonn dataset


Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3


In [5]:
conv1dae_bonn_experiments

{2: {2: {0.25: {'pr_auc': 0.9885693392305245,
    'f1': 0.7927565392354124,
    'recall': 0.6566666666666666,
    'detection_rate': 0.73,
    'detected_anomalies': 73,
    'missed_anomalies': 27,
    'total_anomalies': 100,
    'mean_latency': np.float64(0.410958904109589),
    'median_latency': np.float64(0.0)},
   0.5: {'pr_auc': 0.9903629353039349,
    'f1': 0.8292553191489361,
    'recall': 0.7086363636363636,
    'detection_rate': 0.81,
    'detected_anomalies': 81,
    'missed_anomalies': 19,
    'total_anomalies': 100,
    'mean_latency': np.float64(1.1975308641975309),
    'median_latency': np.float64(0.0)},
   0.75: {'pr_auc': 0.9890606390916102,
    'f1': 0.8565144261870876,
    'recall': 0.7522727272727273,
    'detection_rate': 0.87,
    'detected_anomalies': 87,
    'missed_anomalies': 13,
    'total_anomalies': 100,
    'mean_latency': np.float64(2.1839080459770117),
    'median_latency': np.float64(0.0)}},
  4: {0.25: {'pr_auc': 0.9876875787661518,
    'f1': 0.8545751633

In [6]:
conv1dae_bonn_training_params

{2: {2: {0.25: {'alpha': 1.6872700594236814,
    'beta': 0.38521429192297485,
    'gamma': 1.3659969709057025,
    'lr': 0.015751320499779727,
    'epochs': 29},
   0.5: {'alpha': 1.6872700594236814,
    'beta': 0.38521429192297485,
    'gamma': 1.3659969709057025,
    'lr': 0.015751320499779727,
    'epochs': 29},
   0.75: {'alpha': 1.5779972601681014,
    'beta': 0.11742508365045984,
    'gamma': 1.4330880728874675,
    'lr': 0.015930522616241012,
    'epochs': 43}},
  4: {0.25: {'alpha': 1.803772425950719,
    'beta': 0.15115723710618748,
    'gamma': 1.0325257964926398,
    'lr': 0.07902619549708234,
    'epochs': 50},
   0.5: {'alpha': 1.5610191174223895,
    'beta': 0.2485530730333811,
    'gamma': 1.0171942605576092,
    'lr': 0.06586289317583113,
    'epochs': 31},
   0.75: {'alpha': 1.5610191174223895,
    'beta': 0.2485530730333811,
    'gamma': 1.0171942605576092,
    'lr': 0.06586289317583113,
    'epochs': 31}},
  6: {0.25: {'alpha': 1.892587980696507,
    'beta': 0.159902

In [7]:
for latent in conv1dae_bonn_heatmap.keys():
    heatmaps(conv1dae_bonn_heatmap.get(latent), f"Heatmapy anomalii dla Conv1D-AE z rozmiarem latent = {latent} na zbiorze EEG Bonn", f"{latent}_conv1dae_heatmap_bonn")

#### <center>Zbiór MIT-BIH ECG</center>

Dzięki plikom atr mam dostęp, w którym momencie zostało zarejestrowane uderzenie serca. Dzięki temu mogę każdy z segmentów w sekwencjach oznaczać, ale może być wiele etykiet w segmencie. Aby przypisać czy jest outlierem zliczam wystąpienia "N" i pozostałych i porównuje, czego jest więcej.

In [8]:
mit_bih = load_or_cache_mit_bih()

Przygotowany zbiór MIT BIH

Każdy segment posiada etykietę przypisaną na podstawie stosunku liczby uderzeń normalnych do arytmii.

In [9]:
mit_bih_overlaps = [0.25, 0.5, 0.75]
mit_bih_window_sizes = [2, 3, 5]

<center>Eksperymenty dla 1D Conv-AE</center>

In [10]:
conv1dae_mit_bih_experiments, conv1dae_mit_bih_heatmap, conv1dae_mit_bih_training_params = [
    {
        latent: {
            s: {
                o: None for o in mit_bih_overlaps
            } for s in mit_bih_window_sizes
        } for latent in [2, 4]
    }  for _ in range(3)
]

for latent, seconds in conv1dae_mit_bih_experiments.items():
    for second, overlaps in seconds.items():
        for overlap in overlaps.keys():
            torch.cuda.empty_cache()
            X, y, _ = get_mit_bih_segments(mit_bih, second, overlap)

            # Wybieramy te anomalie, które mają najmniej zanieczyszczonych segmentów
            anomaly_ratios = (y == -1).mean(axis=1)
            pseudo_y = np.where(anomaly_ratios < 0.3, 1, -1)

            # Podział na train/test
            X_train, X_test, y_train, y_test = train_test_split_anomaly_sequence(X, y, pseudo_label=pseudo_y)

            # Optymalizacja parametrów Conv1DAE
            conv1dae_study = optimize_conv1dae(
                X=X_train,
                y=y_train,
                latent=latent,
                dataset_name="MIT BIH",
                n_trials=20,
                percentile=99,
                show_optuna_output=False
            )

            conv1dae_params = conv1dae_study.best_params
            conv1dae_train_params = {k: conv1dae_params[k] for k in ["lr", "epochs", "alpha", "beta", "gamma"]}
            conv1dae_loss_params = {k: conv1dae_params[k] for k in ["alpha", "beta", "gamma"]}

            conv1dae_mit_bih_training_params[latent][second][overlap] = conv1dae_params

            # Konwersja z numpy do torch
            X_train, X_test = torch.from_numpy(X_train).to(dtype=torch.float32, device="cuda"), torch.from_numpy(X_test).to(dtype=torch.float32, device="cuda")

            # Autoenkoder
            conv1dae = Conv1DAE(input_dim=1, latent_dim=latent)
            X_train = X_train[:, :, None, :]
            X_test = X_test[:, :, None, :]

            # Trening
            conv1dae_train(
                model=conv1dae,
                data=X_train,
                **conv1dae_train_params
            )

            # Wyznaczenie progu z danych treningowych
            y_train_scores, reconstruction, encoded = detect_anomalies_conv1dae(
                model=conv1dae,
                data=X_train,
                **conv1dae_loss_params
            )
            train_threshold = np.percentile(y_train_scores, 99)

            # Wyznaczenie etykiet -1/1 na zbiorze testowym
            y_test_scores, reconstruction, encoded = detect_anomalies_conv1dae(
                model=conv1dae,
                data=X_test,
                **conv1dae_loss_params
            )
            y_pred = np.where(y_test_scores > train_threshold, -1, 1)

            # Zapisanie metryk
            experiment_basic_metrics = get_basic_metrics(
                y_test.reshape(-1, 1).squeeze(),
                y_pred.reshape(-1, 1).squeeze(),
                y_test_scores.reshape(-1, 1).squeeze()
            )
            conv1dae_mit_bih_test_latencies = latency_to_detection(y_test, y_pred)
            conv1dae_mit_bih_experiments[latent][second][overlap] = {**experiment_basic_metrics, **conv1dae_mit_bih_test_latencies}
            conv1dae_mit_bih_heatmap[latent][second][overlap] = y_test_scores

            # Rekonstrukcja sygnałów
            segments_reconstruction(
                X_test=X_test.cpu().numpy().squeeze(2),
                X_pred=reconstruction,
                y_true=y_test,
                y_pred=y_pred.squeeze(),
                n=2,
                title="",
                save_path=f"{latent}_{second}_{int(overlap * 100)}_reconstruction_analysis_mit_bih.png"
            )

Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3


In [11]:
conv1dae_mit_bih_experiments

{2: {2: {0.25: {'pr_auc': 0.5588217467724739,
    'f1': 0.14493865030674846,
    'recall': 0.07876640966868098,
    'detection_rate': 0.05024424284717376,
    'detected_anomalies': 72,
    'missed_anomalies': 1361,
    'total_anomalies': 1433,
    'mean_latency': np.float64(5.888888888888889),
    'median_latency': np.float64(0.0)},
   0.5: {'pr_auc': 0.5399895353031791,
    'f1': 0.06815605201733911,
    'recall': 0.03553546592489569,
    'detection_rate': 0.02133632790567097,
    'detected_anomalies': 38,
    'missed_anomalies': 1743,
    'total_anomalies': 1781,
    'mean_latency': np.float64(9.578947368421053),
    'median_latency': np.float64(0.5)},
   0.75: {'pr_auc': 0.4083692155309689,
    'f1': 0.018536317451599615,
    'recall': 0.0093782563390066,
    'detection_rate': 0.008292682926829269,
    'detected_anomalies': 17,
    'missed_anomalies': 2033,
    'total_anomalies': 2050,
    'mean_latency': np.float64(33.05882352941177),
    'median_latency': np.float64(0.0)}},
  3: {

In [12]:
conv1dae_mit_bih_training_params

{2: {2: {0.25: {'alpha': 1.6321961521652106,
    'beta': 0.1710273262090216,
    'gamma': 1.1912120878314645,
    'lr': 0.02113655465531589,
    'epochs': 47},
   0.5: {'alpha': 1.5917022549267168,
    'beta': 0.19127267288786132,
    'gamma': 1.262378215816119,
    'lr': 0.007309539835912915,
    'epochs': 32},
   0.75: {'alpha': 1.831261142176991,
    'beta': 0.19351332282682332,
    'gamma': 1.2600340105889054,
    'lr': 0.0123999678368461,
    'epochs': 29}},
  3: {0.25: {'alpha': 1.892587980696507,
    'beta': 0.15990213464750794,
    'gamma': 1.2571172192068059,
    'lr': 0.015304852121831466,
    'epochs': 26},
   0.5: {'alpha': 1.9970884222858574,
    'beta': 0.2675574192337871,
    'gamma': 1.1782649469421098,
    'lr': 0.02993645567006588,
    'epochs': 42},
   0.75: {'alpha': 1.831261142176991,
    'beta': 0.19351332282682332,
    'gamma': 1.2600340105889054,
    'lr': 0.0123999678368461,
    'epochs': 29}},
  5: {0.25: {'alpha': 1.5917022549267168,
    'beta': 0.19127267288

In [13]:
for latent in conv1dae_mit_bih_heatmap.keys():
    heatmaps(conv1dae_mit_bih_heatmap.get(latent), f"Heatmapy anomalii dla Conv1D-AE z rozmiarem latent = {latent} na zbiorze MIT BIH", f"{latent}_conv1dae_heatmap_mit_bih")